# Testing Multilinguality of Qwen model

The main question this notebook tries to answer is:

- Do models in the Qwen3 model family automatically detect the language and respond in that language using the normal Hugging Face platform?

## Set up code
The code in this notebook assumes you are running your experiments in Google Colab.

In [ ]:
# Import required libraries
import os
import torch as t
from google.colab import userdata
from IPython.display import HTML, display
from transformers import AutoTokenizer, AutoModelForCausalLM

# Select model ID
MODEL_ID = "Qwen/Qwen3-8B"

# Detect device
device = t.device(
    "cuda" if t.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")
if device == "cpu":
    print("No GPU detected.")

# Loading HF_TOKEN (make sure to get a Hugging Face token to gain access to Qwen models)
HF_TOKEN = userdata.get("HF_TOKEN") # Change name of token dependent on your colab secrets

# Loading tokenizer
print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

# Loading model
print(f"Loading model")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=t.float16,    # halves memory footprint vs float32
    device_map="auto",  # automatically distributes across available GPUs/CPU
    token=HF_TOKEN
)

# Place model in inference mode
model.eval()

# Verifying model parameters count
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

## Input - generate - display

In [ ]:
# Constructing input message of user
user_message = "How are you today? Please respond in Japanese."
message = [{"role": "user", "content": user_message}] # Change input message here

# Disable thinking (needed for Qwen models)
chat_template_kwargs = {"enable_thinking": False}

# Building prompt using the chat template method
prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True, **chat_template_kwargs)

# Tokenize the input
inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_length = inputs["input_ids"].shape[1]

# Generate a response
with t.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id
    )

# Decode only the newly generated tokens (exclude the input prompt)
new_tokens = output_ids[0][input_length:]
generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

# Display text generated by model in the terminal window
print(generated_text)

## Pretty printing prompts

In [ ]:
# Select colors for background and bubbles
COLORS = {
    "page_bg": "#f4f3f0",
    "card_bg": "#ffffff",
    "border": "#e5e3dd",
    "system_bg": "#f0f0f0",
    "question_bg": "#f0f0f0",
    "response_bg": "#aaf0c9",
    "text": "#1f2128"
}

# Use specific font stack
FONT_STACK = (
    "-apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif"
)

In [ ]:
# Custom functions for displaying prompts using html
def _block_html(bg_color, text, align="left"):
    # basic HTML-escaping so raw text can't break the markup
    text = (
        str(text)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace("\n", "<br>")
    )
    justify = "flex-start" if align == "left" else "flex-end"
    if "Please respond in Japanese." in text:
        text1, text2 = tuple(text.split("Please"))
        text2 = "Please " + text2
        return f"""
        <div style="display:flex; justify-content:{justify}; margin-bottom:60px;">
        <div style="background:{bg_color}; border-radius:12px; padding:20px 24px;
                    max-width:75%;">
            <div style="font-size:16px; line-height:1.6; color:{COLORS['text']};">
                {text1}<b>{text2}</b>
            </div>
        </div>
        </div>
        """
    return f"""
    <div style="display:flex; justify-content:{justify}; margin-bottom:60px;">
      <div style="background:{bg_color}; border-radius:12px; padding:20px 24px;
                  max-width:75%;">
        <div style="font-size:16px; line-height:1.6; color:{COLORS['text']};">
            {text}
        </div>
      </div>
    </div>
    """


def build_card_html(system_prompt=None, question=None, response=None, response_limiter=None, width=900):
    """Returns a full standalone HTML document (string) for the card."""
    blocks = ""
    if system_prompt:
        blocks += _block_html(COLORS["system_bg"], system_prompt, align="left")
    if question:
        blocks += _block_html(COLORS["question_bg"], question, align="left")
    if response:
        if response_limiter:
            if response_limiter[0] != 0:
                blocks += _block_html(COLORS["response_bg"], f"...{response[response_limiter[0]:response_limiter[1]]}...", align="right")
            else:
                blocks += _block_html(COLORS["response_bg"], f"{response[response_limiter[0]:response_limiter[1]]}...", align="right")
        else:
            blocks += _block_html(COLORS["response_bg"], response, align="right")

    return f"""
    <html>
    <head><meta charset="utf-8"></head>
    <body style="margin:0; padding:40px; background:{COLORS['page_bg']}; font-family:{FONT_STACK};">
      <div style="max-width:{width}px; margin:0 auto; background:{COLORS['card_bg']};
                  border:1px solid {COLORS['border']}; border-radius:16px; padding:32px;">
        {blocks}
      </div>
    </body>
    </html>
    """

In [ ]:
display(HTML(build_card_html(system_prompt=None, question=user_message, response=generated_text, width=400)))